In [1]:
from pathlib import Path
import pandas as pd
import zipfile

PROJECT_ROOT = Path(
    r"D:\Big Data Programming Project\Final Assignment"
)

TIMETABLE_INPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "timetable"
)

TIMETABLE_OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "timetable"
)

SERVICE_DATES = [
    "2025-12-26",
    "2025-12-27",
    "2025-12-28"
]

SCNE_AGENCY_ID = "OP220"

TIMETABLE_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

print("Input root:", TIMETABLE_INPUT_ROOT)
print("Output root:", TIMETABLE_OUTPUT_ROOT)
print("Service dates:", SERVICE_DATES)

Input root: D:\Big Data Programming Project\Final Assignment\data\raw\timetable
Output root: D:\Big Data Programming Project\Final Assignment\data\interim\timetable
Service dates: ['2025-12-26', '2025-12-27', '2025-12-28']


In [2]:
def get_active_scne_timetable(service_date):
    date_compact = service_date.replace("-", "")
    day_column = pd.Timestamp(service_date).day_name().lower()

    zip_path = (
        TIMETABLE_INPUT_ROOT
        / service_date
        / f"itm_north_east_gtfs_{date_compact}.zip"
    )

    with zipfile.ZipFile(zip_path, "r") as archive:
        routes = pd.read_csv(
            archive.open("routes.txt"),
            dtype=str
        )

        trips = pd.read_csv(
            archive.open("trips.txt"),
            dtype=str
        )

        calendar = pd.read_csv(
            archive.open("calendar.txt"),
            dtype=str
        )

        calendar_dates = pd.read_csv(
            archive.open("calendar_dates.txt"),
            dtype=str
        )

        stop_times = pd.read_csv(
            archive.open("stop_times.txt"),
            dtype=str
        )

        stops = pd.read_csv(
            archive.open("stops.txt"),
            dtype=str
        )

    # SCNE routes
    scne_route_ids = set(
        routes.loc[
            routes["agency_id"] == SCNE_AGENCY_ID,
            "route_id"
        ]
    )

    # Regular services active on the date
    regular_services = set(
        calendar.loc[
            (calendar["start_date"] <= date_compact)
            & (calendar["end_date"] >= date_compact)
            & (calendar[day_column] == "1"),
            "service_id"
        ]
    )

    # Calendar exceptions
    exceptions = calendar_dates[
        calendar_dates["date"] == date_compact
    ]

    added_services = set(
        exceptions.loc[
            exceptions["exception_type"] == "1",
            "service_id"
        ]
    )

    removed_services = set(
        exceptions.loc[
            exceptions["exception_type"] == "2",
            "service_id"
        ]
    )

    active_services = (
        regular_services
        .union(added_services)
        .difference(removed_services)
    )

    # Active SCNE trips
    active_trips = trips[
        trips["route_id"].isin(scne_route_ids)
        & trips["service_id"].isin(active_services)
    ].copy()

    active_trips["service_date"] = service_date

    # Active stop times
    active_trip_ids = set(active_trips["trip_id"])

    active_stop_times = stop_times[
        stop_times["trip_id"].isin(active_trip_ids)
    ].copy()

    # Active stops
    active_stop_ids = set(active_stop_times["stop_id"])

    active_stops = stops[
        stops["stop_id"].isin(active_stop_ids)
    ].copy()

    return active_trips, active_stop_times, active_stops


for service_date in SERVICE_DATES:
    trips, stop_times, stops = get_active_scne_timetable(
        service_date
    )

    output_folder = (
        TIMETABLE_OUTPUT_ROOT
        / service_date
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    trips.to_csv(
        output_folder / "scne_active_trips.csv",
        index=False
    )

    stop_times.to_csv(
        output_folder / "scne_active_stop_times.csv",
        index=False
    )

    stops.to_csv(
        output_folder / "scne_active_stops.csv",
        index=False
    )

    print(f"\nDate: {service_date}")
    print("Active trips:", len(trips))
    print("Active routes:", trips["route_id"].nunique())
    print("Stop-time rows:", len(stop_times))
    print("Unique stops:", len(stops))


Date: 2025-12-26
Active trips: 933
Active routes: 31
Stop-time rows: 42444
Unique stops: 1771

Date: 2025-12-27
Active trips: 5004
Active routes: 94
Stop-time rows: 220592
Unique stops: 3992

Date: 2025-12-28
Active trips: 3088
Active routes: 78
Stop-time rows: 135017
Unique stops: 3501


In [3]:
for service_date in SERVICE_DATES:
    output_folder = (
        TIMETABLE_OUTPUT_ROOT
        / service_date
    )

    trips = pd.read_csv(
        output_folder / "scne_active_trips.csv",
        dtype=str
    )

    stop_times = pd.read_csv(
        output_folder / "scne_active_stop_times.csv",
        dtype=str
    )

    stops = pd.read_csv(
        output_folder / "scne_active_stops.csv",
        dtype=str
    )

    print(f"\nDate: {service_date}")
    print("Trips:", len(trips))
    print("Stop-time rows:", len(stop_times))
    print("Stops:", len(stops))
    print(
        "Missing stop coordinates:",
        stops["stop_lat"].isna().sum()
        + stops["stop_lon"].isna().sum()
    )
    print(
        "Duplicate trip-stop sequences:",
        stop_times.duplicated(
            subset=["trip_id", "stop_sequence"]
        ).sum()
    )


Date: 2025-12-26
Trips: 933
Stop-time rows: 42444
Stops: 1771
Missing stop coordinates: 0
Duplicate trip-stop sequences: 0

Date: 2025-12-27
Trips: 5004
Stop-time rows: 220592
Stops: 3992
Missing stop coordinates: 0
Duplicate trip-stop sequences: 0

Date: 2025-12-28
Trips: 3088
Stop-time rows: 135017
Stops: 3501
Missing stop coordinates: 0
Duplicate trip-stop sequences: 0
